# Data Engineering for ML — AI Lab Instructor Notebook

*Deep Learning · Medium*



6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Data Engineering for ML — Student Lab

Focus: build a leakage-safe offline feature table from raw event logs (Bronze → Silver → Gold).

## Section 0 — Generate raw data (Bronze)

We will generate: 
- `users`: user metadata
- `events_raw`: raw events with duplicates + late arrivals
- `labels`: (user_id, as_of_ts, y)

The goal is to build point-in-time features for each label row.

In [ ]:
$10

**Why this works.** # Data Engineering for ML — Instructor Lab

Solutions + short rationales. Focus: Bronze → Silver → Gold with point-in-time correctness.

## Section 0 — Generate raw data (Bronze)

Rationale: create a realistic mini-universe of logs with duplicates + late arrivals so the rest of the lab exercises real production failure modes.

## Task 2: Uniqueness + null checks
*Section: Data Quality checks (Silver)*

## Section 1 — Data Quality checks (Silver)

### Task 1.1: Uniqueness + null checks
- `users.user_id` unique
- `events_raw.event_id` not null
- `labels` has no null `user_id`, `as_of_ts`, `y`

### Task 1.2: Referential integrity
Every `events_raw.user_id` must exist in `users.user_id`

### Task 1.3: Range checks
- `price >= 0`
- `duration_s >= 0` when present

**Hints**
- use Series.is_unique, DataFrame.isna().any(), and set operations

In [ ]:
check('users_user_id_unique', users['user_id'].is_unique)
check('events_event_id_non_null', events_raw['event_id'].notna().all())
check('labels_non_null', labels[['user_id','as_of_ts','y']].notna().all().all())

user_set = set(users['user_id'].values.tolist())
check('events_users_known', set(events_raw['user_id'].dropna().astype(int).unique().tolist()).issubset(user_set))

check('price_nonneg', (events_raw['price'] >= 0).all())
check('duration_nonneg_or_null', ((events_raw['duration_s'].isna()) | (events_raw['duration_s'] >= 0)).all())

In [ ]:
# Checks
check('users_user_id_unique', users['user_id'].is_unique)
check('events_event_id_non_null', events_raw['event_id'].notna().all())
check('labels_non_null', labels[['user_id','as_of_ts','y']].notna().all().all())
check('events_users_known', set(events_raw['user_id'].dropna().astype(int).unique().tolist()).issubset(user_set))
check('price_nonneg', (events_raw['price'] >= 0).all())
check('duration_nonneg_or_null', ((events_raw['duration_s'].isna()) | (events_raw['duration_s'] >= 0)).all())

**Why this works.** ## Section 1 — Data Quality (Silver)

Rationale: you want fast, cheap checks that catch 80% of real data issues: uniqueness, nulls, referential integrity, value ranges.

## Task 3: Create `events_silver` with 1 row per `event_id`.
*Section: Dedupe + Late Arrivals (Silver)*

## Section 2 — Dedupe + Late Arrivals (Silver)
Deduplicate by `event_id`, keeping the row with the latest `ingest_ts`.

### Task 2.1
Create `events_silver` with 1 row per `event_id`.

### Task 2.2
Show (with a small check/print) that duplicates exist in `events_raw` and are removed.

**Hints**
- sort by ingest_ts then drop_duplicates(keep='last')

In [ ]:
raw_dups = events_raw['event_id'].duplicated().sum()
print('raw duplicate rows by event_id:', int(raw_dups))

events_silver = (
    events_raw.sort_values(['event_id','ingest_ts'])
    .drop_duplicates(subset=['event_id'], keep='last')
    .reset_index(drop=True)
)

check('silver_unique_event_id', events_silver['event_id'].is_unique)
print('events_silver rows:', len(events_silver))

In [ ]:
# Checks
check('silver_unique_event_id', events_silver['event_id'].is_unique)

**Why this works.** ## Section 2 — Dedupe + Late Arrivals (Silver)

Rationale: retries create duplicates. The only stable dedupe is a deterministic rule (keep latest ingest_ts) and a stable key (event_id).

## Task 4: Create `daily_user_agg` with columns:
*Section: Incremental daily aggregates (Gold)*

## Section 3 — Incremental daily aggregates (Gold)
We will build daily partitions keyed by `event_date` to simulate incremental processing.

### Task 3.1
Create `daily_user_agg` with columns:
- `event_date`
- `user_id`
- `events_cnt`
- `purchase_cnt`
- `revenue`

### Task 3.2
Implement it so it can be recomputed per-day (partitioned).

In [ ]:
events_silver2 = events_silver.copy()
events_silver2['event_date'] = pd.to_datetime(events_silver2['event_ts']).dt.floor('D')

g = events_silver2.groupby(['event_date','user_id'], as_index=False)
daily_user_agg = g.agg(
    events_cnt=('event_id','count'),
    purchase_cnt=('event_type', lambda s: int((s == 'purchase').sum())),
    revenue=('price','sum'),
)

check('daily_has_cols', set(['event_date','user_id','events_cnt','purchase_cnt','revenue']).issubset(daily_user_agg.columns))
print(daily_user_agg.head())

In [ ]:
# Checks
check('daily_has_cols', set(['event_date','user_id','events_cnt','purchase_cnt','revenue']).issubset(daily_user_agg.columns))

**Why this works.** ## Section 3 — Incremental daily aggregates (Gold)

Rationale: daily partitions are the simplest mental model for incremental computation and backfills: recompute a day, not the world.

## Task 5: Compute 7-day rolling features (as of label time):
*Section: Point-in-time feature table (Gold)*

## Section 4 — Point-in-time feature table (Gold)
The label table has an `as_of_ts`. For each label row, compute features using only days with `event_date <= floor(as_of_ts)`.

### Task 4.1
Compute 7-day rolling features (as of label time):
- `events_7d`
- `purchases_7d`
- `revenue_7d`

### Task 4.2
Demonstrate why a naive global aggregate is leaky by comparing to a leaky feature.

**Hints**
- for each label row, filter daily_user_agg to dates in (as_of_date-6d .. as_of_date) for the same user.

In [ ]:
$12

In [ ]:
# Checks
check('no_negative_counts', (labels2['events_7d'] >= 0).all() and (labels2['purchases_7d'] >= 0).all())

**Why this works.** ## Section 4 — Point-in-time feature table (Gold)

Rationale: computing features from *all* history (including after as_of_ts) is silent label leakage. We want features computed with `event_ts <= as_of_ts`.

Implementation note: we approximate point-in-time joins at day resolution via cumulative sums and as-of merges.

## Task 6: Build `X` from:
*Section: Train-ready baseline*

## Section 5 — Train-ready baseline
Train a baseline classifier and report ROC-AUC with CV.

### Task 5.1
Build `X` from:
- user attributes: `country`, `device_type`
- point-in-time features: `events_7d`, `purchases_7d`, `revenue_7d`

### Task 5.2
Compare ROC-AUC with and without the leaky feature.

In [ ]:
$14

**Why this works.** ## Section 5 — Train-ready baseline

Rationale: your data pipeline should produce features that are immediately model-consumable (types, missingness, encoding). We also show how leakage inflates metrics.